<a href="https://colab.research.google.com/github/bcramp/ML/blob/main/Midterm/Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

Midterm

Title: Midterm Analysis

Name: Brennen Cramp

Date: 10/9/2026

---

# Analysis
---

1. I did not encounter any challenges when claiming my Google Cloud Credits but I did encounter many obstacles when I began provisioning and executing the machine learning workflows in Google Cloud simply because I am foreign to this platform and have not had experience in it before. I had issues when creating my notebook and which account it was looking at (personal account versus SJU account), trouble with finding billing and credit reports, stopping the runtimes to ensure no additional charges were added to my account, and so on. However, for the actual execution once all the preliminary steps were done and I was able to start coding the assignment, I did not have issues besides the pressure of using all of my credits before the allotted amount was utilized. At the end of the assingment, I ran into an issue of thinking I had used all my credits since my runtime terminated but this was due to my internet connection going out for roughly ten minutes. When comparing the cloud environment to local or standard free-tier notebooks, the need of a strong and consistent internet connection was the biggest factor here. I forgot that I was coding the assignment in a Google Cloud Platform (GCP - Vertex AI) and needed the internet to save my code and remain on a runtime, as opposed to a local notebook in Coogle Colab which uses my own machine's resources so I can always be connected and execute the workflows. Finally, I had additional trouble in Task 2 when loading the URL ZIP file into memory since there were two files in the download which required extra steps to parse the folder to get the CSV file.

2. I would usaully determine which column required missing value imputations by using "df.isnull().sum()" (in Task 4.1) from the Pandas library which would normally help count the missing values within the CSV file for numerical features and "df['cat_feature_name'].value_counts()" for categorical features. However, "df.isnull().sum()" did not return any values that were null but also knew that the site that contained the data file (https://archive.ics.uci.edu/dataset/360/air+quality) stated "Missing values are tagged with -200 value." Re-examining the previous task (Task 3 using df.describe()) showed me the -200 sentinel value in the dataset and also showed that what appeared to be numerical columns were actually 'Object' columns since the dataset was using commas as decimals (explaining why the CSV file was semicolon-delimited) for numerical values. To combat this, I replaced all the -200 values with NaN by using .replace(-200, np.nan) which forced the DataFrame to treat the placeholders as actual missing values and would later, in Task 5, update the values that contained commas to decimals (being concious of the 'Date' and 'Time' features) and coerce the 'Object' features to numeric features. From there, a Scikit-Learn SimpleImputer using a median strategy was built into the feature pipeline to replace the NaN entries and using a most_frequent strategy for the categorical features ('Date' and 'Time'). The imputation strategies affected model stability in that if the -200 values were left unhandled, scaling techniques like StandardScaler would treat them as legitimate and highly skewed outliers where this would heavily distort the computed mean and variance. For the numerical features, imputing with the median isolates the core distribution of the atmospheric sensors, ensuring that gradient-based optimization steps do not experience gradient "explosions" or bias toward the negative sentinel values.

3. Scikit-Learn's ColumnTransformer and Pipeline streamline the data preparation process by providing a simultaneous and unified execution since ColumnTransformer automates the manual steps of applying separate transformations (StandardScaler, MinMaxScaler and OneHotEncoder) and combining the dataframes back together. Further, a ColumnTransformer also isolates different feature types, applying numeric scalers only to the physical sensor/meteorological arrays while passing the categorical columns to the OneHotEncoder. The pipeline then links these transformers to the estimator, turning raw data, imputations, scaling, and training into a single execution. For data leakage, this occurs when statistics from the test dataset (such as the global mean or the bin boundaries) influence the training phase. By combining operations into a pipeline and splitting the data before fitting, the transformer calculates parameters strictly from the training set. When evaluating, the pipeline applies those exact training-derived metrics to the test data using the .transform() method, ensuring the test data remains entirely unseen during training, thus preventing any data leakage between training and testing phases.

4. For gradient descent optimization, StandardScaler heavily outperformed MinMaxScaler across the classification tasks. When using the StandardScaler pipeline, softmax accurately hit 97.55% in 2.3351 seconds and SGD accurately hit 97.39% in 0.8462 seconds. In comparison, when I switched to the MinMaxScaler pipeline, performance dropped across the board where softmax accurately hit 92.55% in 2.9884 seconds and SGD accurately hit 93.05% in 0.9671 seconds. This happened because StandardScaler centered the features to a mean of 0 and a variance of 1 which allowed gradients to go directly toward the global minimum. In contrast, MinMaxScaler bounds values between 0 and 1 and if a sensor contains extreme spikes, it compresses the remaining data into a small fraction which flattens the gradients and most likely forced poor decision boundaries. After executing the code blocks, it was seen that optimization algorithms like SGD converge noticeably faster than batch-based methods. The training for SGD with StandardScaler took 0.8462 seconds compared to 2.3351 seconds for softmax with StandardScaler to train (1.4889 seconds slower). The training for SGD with MinMaxScaler took 0.9671 seconds compared to 2.9884 seconds for softmax with MinMaxScaler to train (2.0213 seconds slower). This illustrates SGDClassifier's incredible training speed advantage despite the massive number of one-hot encoded columns. Finally, for evaluation metrics, the baseline linear regression produced exceptional R2 scores of 0.9891 and RMSEs of 0.7921 across both StandardScaler and MinMaxScaler. The exact production of both of these scores proves that standard/unregularized linear models are mathematically invariant to linear shifts in feature scaling. StandardScaler also maintained a high and balanced F1-score of 0.96 for middle-tier pollution categories while MinMaxScaler's F1-score dropped to 0.88 for softmax and 0.89 for SGD. THis can be attributed to MinMaxScaler's scaling of features based on the absolute min and max values where any big pollution spikes would compress the operating data where it can flatten the variance when trying to find subtle differences between the "Low", "Moderate", and "High" air quality categories.

5. Configuring SGDClassifer with loss='modified_hub' brings a hinge loss that is highly tolerant to outlier sensor spikes as opposed to standard closed-form solvers like the "Limited-memory Broyden-Fletcher-Goldfarb-Shanno" (L-BFGS) batch algorithm I used in the assignment. Standard logistic regression relies on batch algorithms that require looking at the entire dataset simultaneously to compare gradients while SGD calculates the gradient iteratively using one sample at a time. This optimization feature makes SGD drastically faster and computationlly easier which helped to cut the training times by roughly 64% (difference of 1.4889 seconds). By processing data iteratively rather than globally, the SGD optimization loop cut training time by almost two-thirds, proving its immense scalability for massive environmental data arrays. Standard SGD hinge loss only provides binary classification boundaries which do not offer nuance when reqgarding the certainty of decisions. When analyzing the data for air quality, this is crucial because instead of a stern and clear "good" or "bad" prediction, the weather system that might be using this data can report a probability percentage of high Benzene (C6H6) toxicity if people were to go outside. Relying only on an overall accuracy metric can hide critical operational flaws if the classes are unbalanced from the beginning and utilizing class-specific metrics from the output evaluations can help with deeper diagnosis. For instance, precision ensures that when the system would issue a high pollutant alert, it is accurate (0.99 for both of the models) which in the context of the real world, would prevent fatigue of warnings since the model is very precise. For recall, it would measure the system's ability to pinpoint/capture each poor air quality instance. The high recall observed for the High-class (0.98 and 0.99) would guarantee that critical toxic spikes are not missed. Finally, for F1-score, this harmonic mean of precision and recall balances both criteria which proves that the models remain robust, thoroughly trained, and stable across all levels of air pollution.

6. Regularization penalties like alpha=10 or alpha=0.1 apply to a fixed numberical contraint on the magnitude of the weights. When MinMaxScaler compressed feature domains, the base coefficients will grow to compensate (like how the PT08.S2 weight spiked to 48.62). Based on the regularized regression models [Ridge (L2) versus Lasso (L1)], as the penalty hyperparameter (alpha = 10) was applied, Ridge regressions dampens collinearity by shrinking feature weights smoothly and keeping every column active and maintaining a sepctacular R2-score of 0.9834 and RMSE of 0.9760 for StandardScaler. In contrast, under MinMaxScaler, Ridge's error increased to an RMSE of 1.3162 and dropping the R2-score to 0.9699. Ridge scales feature weights downward but retains every sensor column in its continuous predictions while Lasso applies an absolute magnitude penalty. However, Lasso regression (alpha = 0.1) utilizes an L1 absolute penalty that forced less relevant/important feature coefficients to hit 0 which the feature selection tool would act aggressively. This converts the underlying dataset into a compressed, sparse format, acting as a built-in automated feature selection tool. For StandardScaler, Lasso completely eliminated 407 features from the model, forcing their coefficients to exactly 0 while for MinMaxScaler, Lasso eliminated 410 features. While Lasso discarded over 400 features for both techniques, it managed to keep a stable high predictability (StandardScaler's R2-score = 0.9681 and RMSE = 1.3545; MinMaxScaler's R2-score = 0.9530 and RMSE = 1.6446). This small drop in predicatability when comparing to the huge compression shows that the majority of the dataset was very redundant and not that important, at least for predicting the 'C6H6' target. In addition, there was a dominant environmental predictor in the 'PT08.S2(NMHC)' feature where it emerged as the single most dominant predictor of the air containing Benzene across all models. For StandardScaler's linear regression, it scored a high positive weight of 7.0578 and despite Lasso pruning above 400 columns, 'num__PT08.S2(NMHC)' was still high at 7.3785. For MinMaxScaler, linear regression gave this feature the highest weight of 48.6292 and 44.2954 for Lasso. In reference to opeational environmental insights revealed by the coefficient shrinkage, the fact that PT08.S2(NMHC) (i.e. "Non-Methane Hydrocarbon") holds the most dominant weight can inform the operators of the prediction system that when high hourly concentrations of this gas are beginning to surface, the public will need to be informed for poor air quality that can lead to toxicity and whatever causes this gas needs to be reduced swiftly. Operators can also take note to how Lasso completely elimated columns (using MinMaxScaler) like absolute humidity (num_AH), hourly nitrogen oxide (num_PT08.S3(NOx)), and temperature (num_T) to prove that local fluctuations in weather patterns are simply noise when trying to predict for Benzene (C6H6) air pollution risk.